In [ ]:
!pip install librosa pydub OpenAI-Whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
  Created wheel for OpenAI-Whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=9e4e50aef47d728268418fbbe57c0c34ffd4c336c7cde5183dbd8979b2f33ce0
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built OpenAI-Whisper


In [ ]:
!pip install -U openai-whisper pyannote.audio google-cloud-storage

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
!pip install huggingface_hub
from huggingface_hub import login


login("BURAYA_TOKEN_GELECEK")

In [ ]:
# 1. KURULUMLAR
!pip install -U openai-whisper pyannote.audio google-cloud-storage huggingface_hub

import whisper
import datetime
import os
import torch
import re
import gc
from google.cloud import storage
from google.colab import auth
from pyannote.audio import Pipeline
from huggingface_hub import login

# --- AYARLAR ---
PROJECT_ID = 'senior-design-488908'
BUCKET_NAME = 'lectureai_processed'
FOLDER_NAME = 'test_trigger/'
HF_TOKEN = 'BURAYA_TOKEN_GELECEK' # Kendi token  'ın
OUTPUT_TXT = 'KESIN_DERS_DOKUMU.txt'
# ---------------

auth.authenticate_user()
login(HF_TOKEN)

def run_fast_turbo_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 🚀 MODELLERİ BİR KEZ YÜKLE VE RAM'DE TUT (Hızın Sırrı)
    print("🚀 Modeller belleğe alınıyor (Turbo Kalitesi + Işık Hızı)...")
    whisper_model = whisper.load_model("turbo", device=device)
    diar_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
    diar_pipeline.to(device)

    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    blobs = client.list_blobs(BUCKET_NAME, prefix=FOLDER_NAME)
    segment_list = [b.name for b in blobs if 'seg_' in b.name and b.name.endswith('.mp4')]
    segment_list.sort(key=lambda x: int(re.search(r'seg_(\d+)', x).group(1)))

    with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
        for i, blob_name in enumerate(segment_list):
            temp_file = "temp_segment.mp4"
            print(f"\n💎 [{i+1}/{len(segment_list)}] TURBO İLE İŞLENİYOR...")

            blob = bucket.blob(blob_name)
            blob.download_to_filename(temp_file)

            # 1. HASSAS AYRIM (Zehra Hoca & Can)
            
            diarization = diar_pipeline(temp_file, num_speakers=2)

            def get_tracks(obj):
                if hasattr(obj, 'itertracks'): return list(obj.itertracks(yield_label=True))
                for attr in dir(obj):
                    try:
                        val = getattr(obj, attr)
                        if hasattr(val, 'itertracks'): return list(val.itertracks(yield_label=True))
                    except: continue
                return []

            tracks = get_tracks(diarization)

            # 2. KALİTELİ METİN (Turbo)
            # Modeli silmediğimiz için burası da direkt başlayacak
            result = whisper_model.transcribe(temp_file, language="tr")

            # 3. HOCA TESPİTİ VE YAZIM
            durations = {}
            for turn, _, label in tracks:
                durations[label] = durations.get(label, 0) + (turn.end - turn.start)
            main_spk = max(durations, key=durations.get) if durations else "SPEAKER_00"

            time_offset = i * 600
            for segment in result['segments']:
                start_time = segment['start']
                text = segment['text'].strip()

                curr_spk = "Bilinmeyen"
                for turn, _, label in tracks:
                    if turn.start <= start_time <= turn.end:
                        curr_spk = label
                        break

                label = "HOCA" if curr_spk == main_spk else "ÖĞRENCİ"
                start_fmt = str(datetime.timedelta(seconds=int(start_time + time_offset)))
                print(f"   [{start_fmt}] {label}: {text}")
                f.write(f"[{start_fmt}] {label}: {text}\n")

            os.remove(temp_file)

    print(f"\n🔥 BAŞARDIK! Kaliteli döküm 15 dakikada bitti: {OUTPUT_TXT}")

run_fast_turbo_pipeline()

🚀 Modeller belleğe alınıyor (Turbo Kalitesi + Işık Hızı)...

💎 [1/12] TURBO İLE İŞLENİYOR...


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


   [0:00:00] ÖĞRENCİ: Altyazı M.K.
   [0:00:30] ÖĞRENCİ: Altyazı M.K.
   [0:01:00] ÖĞRENCİ: Altyazı M.K.
   [0:01:30] ÖĞRENCİ: Altyazı M.K.
   [0:01:59] ÖĞRENCİ: Altyazı M.K.
   [0:02:01] HOCA: Altyazı M.K.
   [0:02:03] ÖĞRENCİ: Altyazı M.K.
   [0:02:05] HOCA: Altyazı M.K.
   [0:02:07] ÖĞRENCİ: Altyazı M.K.
   [0:02:37] ÖĞRENCİ: Altyazı M.K.
   [0:02:39] HOCA: Altyazı M.K.
   [0:02:41] HOCA: Altyazı M.K.
   [0:02:43] ÖĞRENCİ: Altyazı M.K.
   [0:02:45] ÖĞRENCİ: Altyazı M.K.
   [0:02:47] ÖĞRENCİ: geldi şimdi oldu bir tane. Oldu mu?
   [0:02:51] ÖĞRENCİ: Oldu şimdi.
   [0:02:52] ÖĞRENCİ: Başka bir hesaptan oldu ama
   [0:02:53] ÖĞRENCİ: kendi hesabım değil. Anladım.
   [0:02:55] ÖĞRENCİ: Çok fazla istek gönderdiğinde de öyle bir sorun
   [0:02:57] HOCA: çıkıyor olabilir.
   [0:03:00] ÖĞRENCİ: Yok bir kere istek gönderdim.
   [0:03:02] ÖĞRENCİ: Bir kere istek gönderdim.
   [0:03:03] ÖĞRENCİ: İkinci de doğruya girdi.
   [0:03:05] ÖĞRENCİ: Yok iki kere.
   [0:03:07] ÖĞRENCİ: Üç kere dört ker